# Deepfake Audio Detection
Classifies speech recordings as **Genuine (Human)** or **Deepfake (AI-Generated)** using MFCC features and a CNN.

Dataset: [The Fake-or-Real Dataset](https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset)

## 1. Install Dependencies

In [ ]:
!pip install librosa scikit-learn torch torchvision torchaudio matplotlib seaborn -q

## 2. Imports

In [ ]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 3. Feature Extraction
Extract 40 MFCC coefficients from each audio file, padded/truncated to a fixed length.

In [ ]:
N_MFCC = 40
MAX_LEN = 200  # fixed time steps
SR = 16000     # sample rate

def extract_mfcc(file_path, n_mfcc=N_MFCC, max_len=MAX_LEN, sr=SR):
    try:
        audio, _ = librosa.load(file_path, sr=sr, duration=4.0)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        # Pad or truncate to fixed length
        if mfcc.shape[1] < max_len:
            mfcc = np.pad(mfcc, ((0, 0), (0, max_len - mfcc.shape[1])), mode='constant')
        else:
            mfcc = mfcc[:, :max_len]
        return mfcc  # shape: (40, 200)
    except Exception as e:
        print(f'Error loading {file_path}: {e}')
        return None

## 4. Load Dataset
Expects the Kaggle dataset unzipped. Folder structure: `for-norm/training/real/` and `for-norm/training/fake/`

In [ ]:
# Update this path to wherever you extracted the dataset
DATA_DIR = '/kaggle/input/the-fake-or-real-dataset/for-norm/training'

def load_dataset(data_dir):
    X, y = [], []
    for label, folder in enumerate(['fake', 'real']):
        folder_path = os.path.join(data_dir, folder)
        files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
        print(f'Loading {len(files)} files from {folder}...')
        for fname in files:
            fpath = os.path.join(folder_path, fname)
            mfcc = extract_mfcc(fpath)
            if mfcc is not None:
                X.append(mfcc)
                y.append(label)  # 0 = fake, 1 = real
    return np.array(X), np.array(y)

X, y = load_dataset(DATA_DIR)
print(f'\nDataset shape: {X.shape}')
print(f'Class distribution — Fake: {(y==0).sum()}, Real: {(y==1).sum()}')

## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Add channel dimension for CNN: (N, 1, 40, 200)
X_train = X_train[:, np.newaxis, :, :].astype(np.float32)
X_test  = X_test[:, np.newaxis, :, :].astype(np.float32)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 6. PyTorch Dataset & DataLoader

In [ ]:
class AudioDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(AudioDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(AudioDataset(X_test,  y_test),  batch_size=32, shuffle=False)

## 7. CNN Model

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        return self.classifier(self.conv_block(x))

model = AudioCNN().to(device)
print(model)

## 8. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

EPOCHS = 15
train_losses, train_accs = [], []

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (out.argmax(1) == y_batch).sum().item()
        total += len(y_batch)

    scheduler.step()
    acc = correct / total
    train_losses.append(total_loss / len(train_loader))
    train_accs.append(acc)
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.4f}')

## 9. Evaluation

In [ ]:
model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        out = model(X_batch)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()  # prob of 'real'
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())

all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# Metrics
acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average='macro')
print(f'Accuracy : {acc:.4f}')
print(f'Macro F1 : {f1:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=['Fake', 'Real']))

### Equal Error Rate (EER)

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
fnr = 1 - tpr
eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
print(f'Equal Error Rate (EER): {eer*100:.2f}%')
print(f'\nVerification thresholds:')
print(f'  Accuracy >= 80%  : {"PASS" if acc >= 0.80 else "FAIL"}')
print(f'  EER <= 12%       : {"PASS" if eer <= 0.12 else "FAIL"}')
print(f'  Macro F1 >= 80%  : {"PASS" if f1 >= 0.80 else "FAIL"}')

### Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

### Training Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(train_losses, marker='o'); ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch')
ax2.plot(train_accs,   marker='o', color='green'); ax2.set_title('Training Accuracy'); ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_curve.png', dpi=150)
plt.show()

## 10. Save Model

In [ ]:
torch.save(model.state_dict(), 'deepfake_audio_cnn.pth')
print('Model saved to deepfake_audio_cnn.pth')

## 11. Inference on a Single File
Use this function in your Streamlit app.

In [ ]:
def predict(file_path, model, device):
    mfcc = extract_mfcc(file_path)
    if mfcc is None:
        return None, None
    x = torch.tensor(mfcc[np.newaxis, np.newaxis, :, :], dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        out = model(x)
        probs = torch.softmax(out, dim=1).cpu().numpy()[0]
        pred = np.argmax(probs)
    label = 'Real (Human)' if pred == 1 else 'Deepfake (AI-Generated)'
    confidence = probs[pred]
    return label, confidence

# Example usage:
# label, conf = predict('path/to/audio.wav', model, device)
# print(f'{label} | Confidence: {conf:.2%}')